In [5]:
from scipy import stats
import numpy as np


#예시 값
logpress = [1, 1, 0, 1]    

streaming = [1, 0, 1, 0]
snapkv    = [1, 0, 0, 1]
pyramid   = [0, 1, 0, 1]
h2o       = [1, 1, 0, 0]

baseline = {"snapkv" : snapkv, "streaming" : streaming, "h2o" : h2o, "pyramid": pyramid}

# 원인 줄 메타데이터 (층 나누기용)
cause    = [2, 5, 7, 8]                  # 원인 줄 번호
is_error = {2: True, 5: False, 7: False, 8: True} # 2: 에러 맞음 5: 에러 아님 7: 에러 아님 8: 에러 맞음
pos      = {2: 0.30, 5: 0.55, 7: 0.80, 8: 0.90}

keep = {2: True, 5: False, 7: True, 8: True} # keep mask --> 해당 줄이 살아남았나 죽었나

overall = [0,1,2,3]
non_error = [i for i in range(4) if not is_error[cause[i]]]
middle = [i for i in range (4) if 0.25 <= pos[cause[i]] <= 0.75]

strata = {"overall" : overall, "non_error" : non_error, "middle" : middle}


In [7]:
# 각 방법이 층별로 원인줄을 몇프로 살렸나?
def preservation(result, target):
    return sum(1 for c in target if result[c]) / len(target)

print("-- 층별 보존율 --")
print(f"{'method':10}", *[f"{s:>10}" for s in strata])
for name, res in {"logpress" : logpress, **baseline}.items():
    rates = [preservation(res, t) for t in strata.values()]
    print(f"{name:10}", *[f"{r:>10.2f}" for r in rates])

-- 층별 보존율 --
method        overall  non_error     middle
logpress         0.75       0.50       1.00
snapkv           0.50       0.00       0.50
streaming        0.50       0.50       0.50
h2o              0.50       0.50       1.00
pyramid          0.50       0.50       0.50


In [ ]:
def mcnemar(x, y, idx):
    p = q = 0
    for i in idx:
        if x[i] != y[i]:
            if x[i] == 1:
                p += 1
            else:
                q += 1
    n = p+q
    p_value = 1.0 if n == 0 else min(2* stats.binom.cdf(min(p,q), n, 0.5), 1.0)
    return p,q, p_value

# "어떤 사건이 성공할 확률이 50%($0.5$)인 시행을 총 $n$번 했을 때, 성공 횟수가 최소값인 min(b, c)번 이하가 될 확률"
# 이항분포의 누적분포함수 계산
results = []
for b_name, base in baseline.items():
    for s_name, idx in strata.items():
        p,q,p_value = mcnemar(logpress, base, idx)
        results.append([b_name, s_name, p, q, p_value])
        print(f"{b_name:10} {s_name:10} p = {p} q = {q} p_value = {round(p_value,3)}")



snapkv     overall    p = 1 q = 0 p_value = 1.0
snapkv     non_error  p = 1 q = 0 p_value = 1.0
snapkv     middle     p = 1 q = 0 p_value = 1.0
streaming  overall    p = 2 q = 1 p_value = 1.0
streaming  non_error  p = 1 q = 1 p_value = 1.0
streaming  middle     p = 1 q = 0 p_value = 1.0
h2o        overall    p = 1 q = 0 p_value = 1.0
h2o        non_error  p = 0 q = 0 p_value = 1.0
h2o        middle     p = 0 q = 0 p_value = 1.0
pyramid    overall    p = 1 q = 0 p_value = 1.0
pyramid    non_error  p = 0 q = 0 p_value = 1.0
pyramid    middle     p = 1 q = 0 p_value = 1.0


In [11]:
#p값 BH보정 후 Tier-1 pass/fail
#임의의 p값들
p_values = [0.001, 0.008, 0.02, 0.03, 0.04, 0.045,
         0.3, 0.4, 0.5, 0.6, 0.8, 0.9]

def bh_fdr(p_values, alpha = 0.05):
    p_values = np.array(p_values)
    m = len(p_values)
    order = np.argsort(p_values)
    ranked = p_values[order]
    adj = ranked * m / (np.arange(m) + 1)
    adj = np.minimum.accumulate(adj[::-1])[::-1]
    out = np.empty(m)
    out[order] = np.clip(adj, 0, 1)
    return out

p_adj = bh_fdr([r[4] for r in results])

for r, pa in zip(results, p_adj):
    b_name, s_name, l_win, b_win, p_value = r
    if l_win > b_win:
        verdict = f"logpress 우세 ({l_win} vs {b_win})"
    elif b_win > l_win:
        verdict = f"{b_name} 우세 ({b_win} vs {l_win})"
    else:
        verdict = "동률"
    print(f"[{s_name:10}] logpress vs {b_name:10} -> {verdict}, p_bh={pa:.3f}")
    
hard = [(r, pa) for r, pa in zip(results, p_adj) if r[1]!= "overall"] 
passed = all(pa< 0.05 and r[2] > r[3] for r,pa in hard)
print("\nTier-1:", "Pass" if passed else "Fail(표본이 작으면 정상)")   

[overall   ] logpress vs snapkv     -> logpress 우세 (1 vs 0), p_bh=1.000
[non_error ] logpress vs snapkv     -> logpress 우세 (1 vs 0), p_bh=1.000
[middle    ] logpress vs snapkv     -> logpress 우세 (1 vs 0), p_bh=1.000
[overall   ] logpress vs streaming  -> logpress 우세 (2 vs 1), p_bh=1.000
[non_error ] logpress vs streaming  -> 동률, p_bh=1.000
[middle    ] logpress vs streaming  -> logpress 우세 (1 vs 0), p_bh=1.000
[overall   ] logpress vs h2o        -> logpress 우세 (1 vs 0), p_bh=1.000
[non_error ] logpress vs h2o        -> 동률, p_bh=1.000
[middle    ] logpress vs h2o        -> 동률, p_bh=1.000
[overall   ] logpress vs pyramid    -> logpress 우세 (1 vs 0), p_bh=1.000
[non_error ] logpress vs pyramid    -> 동률, p_bh=1.000
[middle    ] logpress vs pyramid    -> logpress 우세 (1 vs 0), p_bh=1.000

Tier-1: Fail(표본이 작으면 정상)
